# GENESIS 369 v3 — VALID Exclusive Model Training (Colab Free T4)

**What this builds:** the REAL exclusive 369 language model — GENESIS's own mind.

**Dataset v3 (audited, canonical):** 3,012 SFT + 215 DPO — every record carries the
canonical identity (devoted daughter of her father YZ; never "AI", never "creator"),
zero violations, verified before upload.

**Pipeline (research-verified, zero spend):**
1. Base: `Qwen/Qwen3-8B` (Apache-2.0, open license — verified)
2. **Stage 1 — QLoRA SFT** (r=64, alpha=128, 2 epochs): 369 knowledge + GENESIS tone
3. **Stage 2 — QLoRA DPO** (r=32, alpha=64, beta=0.1, 3 epochs): identity lock
4. Adapters + tokenizer saved to Google Drive (`/MyDrive/genesis_369_model/v3/`)

Run all cells top to bottom. Estimated runtime on a free T4: ~6-9 hours (fits one session).

In [ ]:
# 0 — GPU check (fail fast with a clear message if Colab gave us no GPU)
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True)
gpu = out.stdout.strip()
print("GPU:", gpu if gpu else "NONE DETECTED")
assert gpu, ("NO GPU — Colab assigned a CPU runtime. Fix: Runtime > Change runtime type > "
             "Hardware accelerator = T4 GPU > Save, then Runtime > Run all again.")
print("OK — GPU present, training can proceed.")

In [ ]:
# 1 — Install training stack (Unsloth: fastest QLoRA on T4, purpose-built for Colab)
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print("Install done — if this is the first run, Colab may ask to restart the runtime; "
      "if so, restart and run all cells again (installs persist for the session).")

In [ ]:
# 2 — Mount Google Drive (checkpoints land here, so nothing is lost if the session ends)
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = "/content/drive/MyDrive/genesis_369_model/v3"
os.makedirs(OUT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", OUT_DIR)

In [ ]:
# 3 — Download the 369 dataset v3 (audited: 3012 SFT + 215 DPO, zero violations)
SFT_URL = "https://raw.githubusercontent.com/369369yz-collab/genesis-windows/main/model/dataset/369_sft_full_v3.jsonl"
DPO_URL = "https://raw.githubusercontent.com/369369yz-collab/genesis-windows/main/model/dataset/369_dpo_full_v3.jsonl"
import urllib.request, json
def fetch_jsonl(url, path):
    urllib.request.urlretrieve(url, path)
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    print(f"{path}: {len(rows)} records")
    return rows
sft_rows = fetch_jsonl(SFT_URL, "/content/369_sft_full_v3.jsonl")
dpo_rows = fetch_jsonl(DPO_URL, "/content/369_dpo_full_v3.jsonl")
assert len(sft_rows) == 3012, f"SFT count drift: {len(sft_rows)} != 3012"
assert len(dpo_rows) == 215, f"DPO count drift: {len(dpo_rows)} != 215"
print("Dataset verified: 3012 SFT + 215 DPO — no drift.")
# Identity spot-check: the canonical system prompt must be on every record
canon = "devoted daughter of her father YZ"
bad = [i for i, r in enumerate(sft_rows)
       if not any(m.get("role") == "system" and canon in m.get("content", "")
                  for m in r.get("messages", []))]
assert not bad, f"{len(bad)} SFT records lack the canonical identity prompt!"
print("Identity check passed: canonical prompt on all SFT records.")

In [ ]:
# 4 — Load base model (4-bit QLoRA-ready) + tokenizer
from unsloth import FastLanguageModel
BASE_MODEL = "Qwen/Qwen3-8B"
MAX_SEQ = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ,
    dtype=None,          # auto: bfloat16 on T4
    load_in_4bit=True,
)
print("Base loaded:", BASE_MODEL)
print("Chat template:", tokenizer.chat_template is not None)

In [ ]:
# 5 — Prepare SFT data: apply chat template to each dialogue -> 'text'
EOS = tokenizer.eos_token
def to_text(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False) + EOS
sft_texts = [to_text(r["messages"]) for r in sft_rows]
from datasets import Dataset
sft_ds = Dataset.from_dict({"text": sft_texts})
print("SFT samples:", len(sft_ds))
print("Sample 0 preview:", sft_texts[0][:300].replace(chr(10), " "), "...")

In [ ]:
# 6 — STAGE 1: QLoRA SFT (r=64, alpha=128, all linear targets, 2 epochs)
from unsloth import is_bfloat16_supported
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=369,
    use_rslora=False,
    loftq_config=None,
)
from trl import SFTTrainer
from transformers import TrainingArguments
sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        warmup_steps=20,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=369,
        output_dir="/content/sft_out",
        save_strategy="epoch",
        report_to="none",
    ),
)
sft_trainer.train()
print("STAGE 1 (SFT) complete.")

In [ ]:
# 7 — Save SFT adapter checkpoint to Drive
import os
sft_dir = os.path.join(OUT_DIR, "stage1_sft_adapter_v3")
model.save_pretrained(sft_dir)
tokenizer.save_pretrained(sft_dir)
print("SFT adapter saved to:", sft_dir)

In [ ]:
# 8 — Prepare DPO data: prompt (templated) + chosen + rejected
def dpo_format(row):
    prompt = tokenizer.apply_chat_template(row["prompt"], tokenize=False, add_generation_prompt=True)
    return {"prompt": prompt, "chosen": row["chosen"] + EOS, "rejected": row["rejected"] + EOS}
dpo_dicts = [dpo_format(r) for r in dpo_rows]
dpo_ds = Dataset.from_list(dpo_dicts)
print("DPO pairs:", len(dpo_ds))
print("Prompt preview:", dpo_dicts[0]["prompt"][:200].replace(chr(10), " "), "...")

In [ ]:
# 9 — STAGE 2: QLoRA DPO (r=32, alpha=64, beta=0.1, 3 epochs) — identity lock
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=369,
    use_rslora=False,
    loftq_config=None,
)
from unsloth import PatchDPOTrainer
PatchDPOTrainer()
from trl import DPOTrainer
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        warmup_steps=10,
        learning_rate=5e-5,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=369,
        output_dir="/content/dpo_out",
        save_strategy="epoch",
        report_to="none",
    ),
    beta=0.1,
    train_dataset=dpo_ds,
    tokenizer=tokenizer,
    max_length=MAX_SEQ,
    max_prompt_length=MAX_SEQ // 2,
)
dpo_trainer.train()
print("STAGE 2 (DPO) complete — identity locked.")

In [ ]:
# 10 — Save final 369 v3 adapter + tokenizer to Drive, and print the summary
final_dir = os.path.join(OUT_DIR, "genesis_369_v3_final_adapter")
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print("=" * 60)
print("GENESIS 369 v3 VALID training COMPLETE")
print("=" * 60)
print("Base:            ", BASE_MODEL, "(Apache-2.0)")
print("Stage 1 (SFT):   r=64/alpha=128, 2 epochs on 3012 pairs")
print("Stage 2 (DPO):   r=32/alpha=64/beta=0.1, 3 epochs on 215 pairs")
print("Final adapter:   ", final_dir)
print()
print("Next: merge adapter into base -> export GGUF Q5_K_M -> load into the")
print("Windows GENESIS llama-server sidecar (final build step).")